# kg-gen을 활용한 한국 농업 지식그래프 생성

이 노트북은 kg-gen의 공식 문서에 따라 한국 농업 텍스트 데이터에서 지식그래프를 생성합니다.

## 파이프라인 개요
1. **개별 문서 처리**: 각 문서를 독립적으로 처리 (Rate limit 대응)
2. **그래프 통합**: 개별 그래프들을 하나로 통합
3. **클러스터링**: 중복 엔티티/관계 제거 및 정리
4. **시각화 및 분석**: 결과 시각화 및 통계 분석

## 디렉토리 구조
```
kg_output/
├── graphs/
│   ├── individual/     # 개별 문서 그래프
│   ├── aggregated/     # 통합된 그래프
│   └── clustered/      # 최종 클러스터링 그래프
├── visualizations/     # HTML 시각화
└── statistics/         # 통계 및 분석
```

## 1. 환경 설정

In [ ]:
# 필요 라이브러리 임포트
import json
import os
import time
from datetime import datetime
from pathlib import Path
import getpass
from collections import Counter
from typing import List, Dict, Any

# kg-gen 라이브러리
try:
    from kg_gen import KGGen
    from kg_gen.models import Graph
    print("✅ kg-gen 라이브러리가 성공적으로 임포트되었습니다.")
except ImportError:
    print("❌ kg-gen이 설치되지 않았습니다. 다음 명령어로 설치하세요:")
    print("pip install kg-gen")
    raise

In [ ]:
# 디렉토리 구조 생성 함수
def create_directory_structure(base_path: str = "kg_output") -> Dict[str, str]:
    """표준 디렉토리 구조를 생성하고 경로들을 반환합니다."""
    directories = {
        "base": base_path,
        "graphs": os.path.join(base_path, "graphs"),
        "individual": os.path.join(base_path, "graphs", "individual"),
        "aggregated": os.path.join(base_path, "graphs", "aggregated"),
        "clustered": os.path.join(base_path, "graphs", "clustered"),
        "visualizations": os.path.join(base_path, "visualizations"),
        "statistics": os.path.join(base_path, "statistics")
    }
    
    # 디렉토리 생성
    for name, path in directories.items():
        os.makedirs(path, exist_ok=True)
        print(f"📁 {name}: {os.path.abspath(path)}")
    
    return directories

# 디렉토리 생성
PATHS = create_directory_structure()
print("\n✅ 디렉토리 구조가 생성되었습니다.")

In [ ]:
# API 키 설정
print("OpenAI API 키를 입력하세요:")
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("✅ API 키가 설정되었습니다.")
else:
    raise ValueError("API 키가 필요합니다.")

## 2. 데이터 준비

In [ ]:
# 농업 데이터 로드
import os

# 현재 노트북의 위치를 기준으로 상대 경로 설정
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
DATA_PATH = os.path.join(notebook_dir, "../rag_dataset/processed_texts/final_texts.json")

# 경로가 존재하지 않으면 절대 경로 사용
if not os.path.exists(DATA_PATH):
    DATA_PATH = "<PROJECT_ROOT>/rag_dataset/processed_texts/final_texts.json"

print(f"데이터 경로: {DATA_PATH}")
print(f"파일 존재 여부: {os.path.exists(DATA_PATH)}")

try:
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        validated_data = json.load(f)
    
    print(f"✅ 농업 데이터 로드 완료!")
    print(f"- 총 문서 수: {len(validated_data)}개")
    
    # 데이터 검증
    total_chars = sum(len(doc['text']) for doc in validated_data)
    avg_chars = total_chars / len(validated_data) if validated_data else 0
    
    print(f"- 전체 텍스트 크기: {total_chars:,} 문자")
    print(f"- 평균 문서 크기: {avg_chars:,.0f} 문자")
    
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다: {DATA_PATH}")
    raise

In [ ]:
# 처리할 문서 수 설정
NUM_DOCS_TO_PROCESS = len(validated_data)  # 테스트: 10개, 전체: len(validated_data)

print(f"🎯 처리할 문서 수: {NUM_DOCS_TO_PROCESS}개")
print("\n📝 샘플 문서 미리보기:")

# 첫 번째 문서 미리보기
sample_doc = validated_data[0]
sample_text = sample_doc['text']
metadata = sample_doc.get('metadata', {})
doc_title = metadata.get('filename', metadata.get('url', '제목 없음'))

print(f"제목: {doc_title}")
print(f"내용: {sample_text[:300]}..." if len(sample_text) > 300 else sample_text)

## 3. KGGen 초기화

In [ ]:
# 기본 KGGen 인스턴스 (개별 문서 처리용)
kg = KGGen(
    model="openai/gpt-4o-mini",  # Rate limit이 더 높은 모델
    temperature=0.0,
    api_key=api_key
)

print("✅ KGGen 초기화 완료")
print(f"- 모델: {kg.model}")
print(f"- 온도: {kg.temperature}")

## 4. 개별 문서 처리

각 문서를 개별적으로 처리하여 Rate limit를 회피하고 안정적으로 실행합니다.

In [ ]:
# 개별 문서 처리 함수
def process_individual_documents(
    documents: List[Dict[str, Any]], 
    kg_instance: KGGen,
    output_dir: str,
    chunk_size: int = 1000,
    delay: float = 1.0
) -> List[Graph]:
    """개별 문서들을 처리하고 그래프를 반환합니다."""
    
    graphs = []
    failed_docs = []
    
    print(f"🔄 {len(documents)}개 문서 처리 시작...")
    start_time = datetime.now()
    
    for i, doc in enumerate(documents):
        doc_text = doc['text']
        metadata = doc.get('metadata', {})
        doc_title = metadata.get('filename', metadata.get('url', f'문서_{i+1}'))
        
        print(f"\n📄 [{i+1}/{len(documents)}] {doc_title[:50]}...")
        
        try:
            # 개별 그래프 생성
            doc_start = datetime.now()
            
            graph = kg_instance.generate(
                input_data=doc_text,
                chunk_size=chunk_size,
                context=f"{doc_title}. 토마토, 고추, 파프리카 등 작물의 생장과 식물 질병, 그리고 정부의 작물 생장을 위한 기술지원 및 문제해결 과정에 초점을 맞춘 한국 농업 문서를 다룬다.",
                output_folder=output_dir  # 각 문서별로 graph.json 저장
            )
            
            # 개별 파일로도 저장 (문서별 추적을 위해)
            individual_path = os.path.join(output_dir, f"doc_{i+1:03d}_graph.json")
            graph_dict = {
                "doc_id": i + 1,
                "doc_title": doc_title,
                "entities": list(graph.entities),
                "relations": [(s, p, o) for s, p, o in graph.relations],
                "edges": list(graph.edges)
            }
            
            with open(individual_path, 'w', encoding='utf-8') as f:
                json.dump(graph_dict, f, ensure_ascii=False, indent=2)
            
            graphs.append(graph)
            
            doc_time = (datetime.now() - doc_start).total_seconds()
            print(f"  ✓ 완료 ({doc_time:.1f}초)")
            print(f"    - 엔티티: {len(graph.entities)}개")
            print(f"    - 관계: {len(graph.relations)}개")
            print(f"    - 저장: {individual_path}")
            
            # Rate limit 회피를 위한 지연
            if i < len(documents) - 1:
                print(f"  ⏳ {delay}초 대기...")
                time.sleep(delay)
                
        except Exception as e:
            print(f"  ✗ 오류: {e}")
            failed_docs.append((i, doc_title, str(e)))
            
            # Rate limit 오류인 경우 추가 대기
            if "rate limit" in str(e).lower():
                print(f"  ⏳ Rate limit 오류 - 5초 추가 대기...")
                time.sleep(5)
    
    # 처리 결과 요약
    total_time = (datetime.now() - start_time).total_seconds()
    print(f"\n✅ 개별 문서 처리 완료!")
    print(f"- 소요 시간: {total_time:.1f}초")
    print(f"- 성공: {len(graphs)}개")
    print(f"- 실패: {len(failed_docs)}개")
    
    if failed_docs:
        print("\n❌ 실패한 문서:")
        for idx, title, error in failed_docs:
            print(f"  - [{idx+1}] {title[:30]}...: {error[:50]}...")
    
    return graphs

In [ ]:
# 개별 문서 처리 실행
documents_to_process = validated_data[:NUM_DOCS_TO_PROCESS]

individual_graphs = process_individual_documents(
    documents=documents_to_process,
    kg_instance=kg,
    output_dir=PATHS["individual"],
    chunk_size=1000,
    delay=1.0
)

## 5. 그래프 통합

개별 그래프들을 하나의 통합 그래프로 합칩니다.

In [ ]:
if individual_graphs:
    print("🔗 개별 그래프 통합 중...")
    
    # 통합 전 통계
    total_entities_before = sum(len(g.entities) for g in individual_graphs)
    total_relations_before = sum(len(g.relations) for g in individual_graphs)
    
    # 그래프 통합
    aggregated_graph = kg.aggregate(individual_graphs)
    
    # 통합된 그래프 저장
    aggregated_path = os.path.join(PATHS["aggregated"], "aggregated_graph.json")
    aggregated_dict = {
        "metadata": {
            "created_at": datetime.now().isoformat(),
            "source_graphs": len(individual_graphs),
            "total_entities_before": total_entities_before,
            "total_relations_before": total_relations_before
        },
        "entities": list(aggregated_graph.entities),
        "relations": [(s, p, o) for s, p, o in aggregated_graph.relations],
        "edges": list(aggregated_graph.edges)
    }
    
    with open(aggregated_path, 'w', encoding='utf-8') as f:
        json.dump(aggregated_dict, f, ensure_ascii=False, indent=2)
    
    print(f"✅ 통합 완료!")
    print(f"\n📊 통합 결과:")
    print(f"- 개별 그래프 수: {len(individual_graphs)}개")
    print(f"- 통합 전 총 엔티티: {total_entities_before:,}개")
    print(f"- 통합 후 엔티티: {len(aggregated_graph.entities):,}개 (중복 제거됨)")
    print(f"- 통합 전 총 관계: {total_relations_before:,}개")
    print(f"- 통합 후 관계: {len(aggregated_graph.relations):,}개")
    print(f"- 저장 위치: {aggregated_path}")
else:
    print("⚠️  통합할 그래프가 없습니다.")
    aggregated_graph = None

## 6. 클러스터링 (선택사항)

유사한 엔티티와 관계를 클러스터링하여 더욱 정제된 그래프를 생성합니다.

In [ ]:
# 클러스터링을 위한 KGGen 인스턴스 (retrieval_model 포함)
try:
    kg_with_retrieval = KGGen(
        model="openai/gpt-4o-mini",
        temperature=0.0,
        api_key=api_key,
        retrieval_model="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    print("✅ Retrieval model이 포함된 KGGen 인스턴스 생성 완료")
    clustering_available = True
except Exception as e:
    print(f"⚠️  Retrieval model 초기화 실패: {e}")
    print("sentence-transformers 설치가 필요할 수 있습니다: pip install sentence-transformers")
    clustering_available = False

In [ ]:
if aggregated_graph and clustering_available:
    print("🎯 통합 그래프에 클러스터링 적용 중...")
    print("⏳ 이 작업은 몇 분 정도 소요될 수 있습니다...")
    
    try:
        cluster_start = datetime.now()
        
        # 프롬프트 파일 전체 로드
        prompt_file_path = '<PROJECT_ROOT>/kg_gen/kg-gen_prompt.txt'
        with open(prompt_file_path, 'r', encoding='utf-8') as f:
            full_prompt = f.read()

        # 클러스터링 실행 - 전체 프롬프트를 context로 전달
        final_graph = kg_with_retrieval.cluster(
            aggregated_graph,
            context=full_prompt  # 전체 프롬프트 사용
        )

        cluster_time = (datetime.now() - cluster_start).total_seconds()
        
        # 클러스터링된 그래프 저장
        clustered_path = os.path.join(PATHS["clustered"], "final_clustered_graph.json")
        clustered_dict = {
            "metadata": {
                "created_at": datetime.now().isoformat(),
                "clustering_time": cluster_time,
                "model": kg_with_retrieval.model,
                "retrieval_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
            },
            "entities": list(final_graph.entities),
            "relations": [(s, p, o) for s, p, o in final_graph.relations],
            "edges": list(final_graph.edges),
            "entity_clusters": {k: list(v) for k, v in final_graph.entity_clusters.items()} if hasattr(final_graph, 'entity_clusters') else {},
            "edge_clusters": {k: list(v) for k, v in final_graph.edge_clusters.items()} if hasattr(final_graph, 'edge_clusters') else {}
        }
        
        with open(clustered_path, 'w', encoding='utf-8') as f:
            json.dump(clustered_dict, f, ensure_ascii=False, indent=2)
        
        print(f"✅ 클러스터링 완료! (소요 시간: {cluster_time:.1f}초)")
        print(f"\n🎯 클러스터링 결과:")
        print(f"- 최종 엔티티: {len(final_graph.entities):,}개")
        print(f"- 최종 관계: {len(final_graph.relations):,}개")
        
        if hasattr(final_graph, 'entity_clusters'):
            print(f"- 엔티티 클러스터: {len(final_graph.entity_clusters)}개")
        if hasattr(final_graph, 'edge_clusters'):
            print(f"- 엣지 클러스터: {len(final_graph.edge_clusters)}개")
        
        print(f"- 저장 위치: {clustered_path}")
        
    except Exception as e:
        print(f"❌ 클러스터링 실패: {e}")
        final_graph = aggregated_graph
else:
    print("⚠️  클러스터링을 건너뜁니다.")
    final_graph = aggregated_graph

## 7. 시각화

In [ ]:
# 시각화 함수
def visualize_graphs(graphs_info: List[tuple], viz_dir: str):
    """여러 그래프를 시각화합니다.
    
    Args:
        graphs_info: [(filename, graph, description), ...] 형태의 리스트
        viz_dir: 시각화 파일을 저장할 디렉토리
    """
    print("🎨 그래프 시각화 생성 중...")
    
    for filename, graph, description in graphs_info:
        if graph is None:
            continue
            
        output_path = os.path.join(viz_dir, filename)
        
        try:
            KGGen.visualize(
                graph, 
                output_path, 
                open_in_browser=False
            )
            print(f"✓ {description}: {output_path}")
        except Exception as e:
            print(f"✗ {description} 시각화 실패: {e}")

In [ ]:
# 시각화할 그래프 준비
graphs_to_visualize = []

# 샘플: 첫 번째 개별 그래프
if individual_graphs:
    graphs_to_visualize.append(
        ("sample_individual_graph.html", individual_graphs[0], "샘플 개별 그래프")
    )

# 통합 그래프
if aggregated_graph:
    graphs_to_visualize.append(
        ("aggregated_graph.html", aggregated_graph, "통합 그래프")
    )

# 최종 클러스터링 그래프
if 'final_graph' in locals() and final_graph:
    graphs_to_visualize.append(
        ("final_clustered_graph.html", final_graph, "최종 클러스터링 그래프")
    )

# 시각화 실행
visualize_graphs(graphs_to_visualize, PATHS["visualizations"])

print("\n📁 브라우저에서 HTML 파일을 열어 인터랙티브 그래프를 확인하세요!")

## 8. 결과 분석 및 통계

In [ ]:
# 분석할 그래프 선택
analysis_graph = final_graph if 'final_graph' in locals() and final_graph else aggregated_graph
graph_name = "최종 클러스터링 그래프" if 'final_graph' in locals() and final_graph else "통합 그래프"

if analysis_graph:
    print(f"📊 {graph_name} 분석")
    print("=" * 60)
    
    # 기본 통계
    print(f"\n🔍 기본 통계:")
    print(f"- 총 엔티티: {len(analysis_graph.entities):,}개")
    print(f"- 총 관계: {len(analysis_graph.relations):,}개")
    print(f"- 고유 관계 타입: {len(analysis_graph.edges):,}개")
    
    # 엔티티 빈도 분석
    entity_frequency = Counter()
    for relation in analysis_graph.relations:
        entity_frequency[relation[0]] += 1
        entity_frequency[relation[2]] += 1
    
    print(f"\n🌟 가장 연결이 많은 엔티티 (상위 15개):")
    for i, (entity, count) in enumerate(entity_frequency.most_common(15), 1):
        print(f"  {i:2d}. {entity}: {count}개 연결")
    
    # 관계 타입 분석
    edge_frequency = Counter([relation[1] for relation in analysis_graph.relations])
    
    print(f"\n🔗 가장 자주 사용되는 관계 타입 (상위 10개):")
    for i, (edge, count) in enumerate(edge_frequency.most_common(10), 1):
        print(f"  {i:2d}. '{edge}': {count}회")
    
    # 클러스터 정보 (있는 경우)
    if hasattr(analysis_graph, 'entity_clusters') and analysis_graph.entity_clusters:
        print(f"\n🎯 클러스터 정보:")
        print(f"- 엔티티 클러스터 수: {len(analysis_graph.entity_clusters)}개")
        
        # 큰 클러스터 예시
        print("\n큰 엔티티 클러스터 예시 (상위 5개):")
        sorted_clusters = sorted(analysis_graph.entity_clusters.items(), 
                               key=lambda x: len(x[1]), reverse=True)[:5]
        for i, (key, cluster) in enumerate(sorted_clusters, 1):
            cluster_list = list(cluster)
            print(f"  {i}. '{key}' ({len(cluster)}개): {', '.join(cluster_list[:3])}{'...' if len(cluster_list) > 3 else ''}")

In [ ]:
# 통계 정보 저장
if analysis_graph:
    statistics = {
        "analysis_time": datetime.now().isoformat(),
        "graph_type": graph_name,
        "source_documents": NUM_DOCS_TO_PROCESS,
        "basic_stats": {
            "total_entities": len(analysis_graph.entities),
            "total_relations": len(analysis_graph.relations),
            "unique_edge_types": len(analysis_graph.edges),
            "entity_clusters": len(getattr(analysis_graph, 'entity_clusters', {})),
            "edge_clusters": len(getattr(analysis_graph, 'edge_clusters', {}))
        },
        "top_entities": dict(entity_frequency.most_common(30)),
        "top_edges": dict(edge_frequency.most_common(20)),
        "processing_info": {
            "model": kg.model,
            "chunk_size": 1000,
            "clustering_enabled": 'final_graph' in locals()
        }
    }
    
    stats_path = os.path.join(PATHS["statistics"], "kg_statistics.json")
    with open(stats_path, 'w', encoding='utf-8') as f:
        json.dump(statistics, f, ensure_ascii=False, indent=2)
    
    print(f"\n📊 통계 정보 저장: {stats_path}")

## 9. 최종 요약

In [ ]:
print("=" * 70)
print("✅ 한국 농업 지식그래프 생성 완료!")
print("=" * 70)

print(f"\n📁 출력 디렉토리: {os.path.abspath(PATHS['base'])}")
print("\n📊 생성된 파일:")

# 각 디렉토리의 파일 수 확인
for dir_name in ['individual', 'aggregated', 'clustered', 'visualizations', 'statistics']:
    dir_path = PATHS[dir_name]
    if os.path.exists(dir_path):
        files = [f for f in os.listdir(dir_path) if os.path.isfile(os.path.join(dir_path, f))]
        if files:
            print(f"\n  {dir_name}/ ({len(files)}개 파일)")
            for file in files[:5]:  # 최대 5개만 표시
                file_path = os.path.join(dir_path, file)
                size = os.path.getsize(file_path)
                print(f"    • {file} ({size:,} bytes)")
            if len(files) > 5:
                print(f"    ... 외 {len(files)-5}개 파일")

if analysis_graph:
    print(f"\n📈 최종 그래프 요약:")
    print(f"- 처리된 문서: {NUM_DOCS_TO_PROCESS}개")
    print(f"- 추출된 엔티티: {len(analysis_graph.entities):,}개")
    print(f"- 추출된 관계: {len(analysis_graph.relations):,}개")
    print(f"- 고유 관계 타입: {len(analysis_graph.edges):,}개")

print("\n🎯 다음 단계:")
print("1. visualizations/ 폴더의 HTML 파일로 그래프 탐색")
print("2. statistics/kg_statistics.json으로 상세 분석")
print("3. 더 많은 문서로 확장 (NUM_DOCS_TO_PROCESS 증가)")
print("4. GraphRAG와 성능 비교")

## 10. 엔티티 임베딩 생성 및 저장

검색 성능 향상을 위해 모든 엔티티에 대한 임베딩을 미리 계산하고 저장합니다.
이는 나중에 쿼리 시 실시간 임베딩 계산을 피하고 빠른 검색을 가능하게 합니다.

In [ ]:
# 임베딩 생성을 위한 라이브러리 import
import numpy as np
from sentence_transformers import SentenceTransformer
import pickle
from typing import Dict, List, Set

# 임베딩 모델 초기화 (kg-gen에서 사용하는 모델과 동일)
print("🚀 임베딩 모델 로드 중...")
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("✅ 임베딩 모델 로드 완료")

def create_entity_embeddings_with_clusters(graph_path: str, output_dir: str) -> Dict[str, Dict]:
    """
    그래프의 모든 엔티티에 대한 임베딩을 생성하고 클러스터 정보와 함께 저장합니다.
    
    Args:
        graph_path: 그래프 JSON 파일 경로
        output_dir: 임베딩 파일을 저장할 디렉토리
        
    Returns:
        엔티티 정보 딕셔너리 (임베딩, 클러스터 정보 포함)
    """
    print(f"\n📊 그래프 로드: {graph_path}")
    
    # 그래프 데이터 로드
    with open(graph_path, 'r', encoding='utf-8') as f:
        graph_data = json.load(f)
    
    entities = graph_data.get('entities', [])
    entity_clusters = graph_data.get('entity_clusters', {})
    relations = graph_data.get('relations', [])
    
    print(f"- 엔티티 수: {len(entities)}")
    print(f"- 엔티티 클러스터 수: {len(entity_clusters)}")
    print(f"- 관계 수: {len(relations)}")
    
    # 엔티티 임베딩 생성
    print("\n🔄 엔티티 임베딩 생성 중...")
    entity_info = {}
    
    # 배치 처리를 위해 엔티티 리스트 준비
    entity_texts = []
    entity_keys = []
    
    for entity in entities:
        # 엔티티가 문자열인 경우 따옴표 제거
        entity_text = entity.strip('"') if isinstance(entity, str) else str(entity)
        entity_texts.append(entity_text)
        entity_keys.append(entity_text)
    
    # 배치 임베딩 생성
    if entity_texts:
        embeddings = embedding_model.encode(entity_texts, show_progress_bar=True)
        
        # 각 엔티티에 대한 정보 저장
        for key, embedding in zip(entity_keys, embeddings):
            entity_info[key] = {
                'embedding': embedding,
                'cluster_members': set(),  # 같은 클러스터의 다른 엔티티들
                'cluster_id': None         # 클러스터 대표 엔티티
            }
    
    # 클러스터 정보 매핑
    print("\n🎯 클러스터 정보 매핑 중...")
    entity_to_cluster = {}  # 엔티티 -> 클러스터 대표 매핑
    
    for cluster_id, members in entity_clusters.items():
        cluster_id_clean = cluster_id.strip('"')
        members_clean = [m.strip('"') for m in members]
        
        # 모든 멤버에 대해 클러스터 정보 저장
        for member in members_clean:
            if member in entity_info:
                entity_info[member]['cluster_id'] = cluster_id_clean
                entity_info[member]['cluster_members'] = set(members_clean) - {member}
                entity_to_cluster[member] = cluster_id_clean
    
    # 1-hop 이웃 정보 계산 (선택사항)
    print("\n🔗 1-hop 이웃 정보 계산 중...")
    entity_neighbors = {entity: {'outgoing': set(), 'incoming': set()} for entity in entity_keys}
    
    for relation in relations:
        if len(relation) >= 3:
            source = relation[0].strip('"') if isinstance(relation[0], str) else str(relation[0])
            target = relation[2].strip('"') if isinstance(relation[2], str) else str(relation[2])
            
            if source in entity_neighbors:
                entity_neighbors[source]['outgoing'].add(target)
            if target in entity_neighbors:
                entity_neighbors[target]['incoming'].add(source)
    
    # 이웃 정보 추가
    for entity, neighbors in entity_neighbors.items():
        if entity in entity_info:
            entity_info[entity]['neighbors'] = neighbors
    
    # 임베딩 데이터만 따로 저장 (빠른 로드를 위해)
    embeddings_only = {k: v['embedding'] for k, v in entity_info.items()}
    embeddings_path = os.path.join(output_dir, 'entity_embeddings.pkl')
    with open(embeddings_path, 'wb') as f:
        pickle.dump(embeddings_only, f)
    
    # 전체 엔티티 정보 저장 (클러스터 정보 포함)
    # numpy 배열을 리스트로 변환하여 JSON 직렬화 가능하게 만들기
    entity_info_serializable = {}
    for k, v in entity_info.items():
        entity_info_serializable[k] = {
            'cluster_id': v['cluster_id'],
            'cluster_members': list(v['cluster_members']),
            'neighbors': {
                'outgoing': list(v['neighbors']['outgoing']),
                'incoming': list(v['neighbors']['incoming'])
            } if 'neighbors' in v else {'outgoing': [], 'incoming': []}
        }
    
    entity_info_path = os.path.join(output_dir, 'entity_info.json')
    with open(entity_info_path, 'w', encoding='utf-8') as f:
        json.dump(entity_info_serializable, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ 임베딩 및 클러스터 정보 생성 완료!")
    print(f"- 생성된 임베딩: {len(entity_info)}개")
    print(f"- 클러스터에 속한 엔티티: {sum(1 for v in entity_info.values() if v['cluster_id'])}개")
    print(f"- 임베딩 파일: {embeddings_path}")
    print(f"- 엔티티 정보: {entity_info_path}")
    
    # 추가 메타데이터 저장
    metadata = {
        "created_at": datetime.now().isoformat(),
        "model": "paraphrase-multilingual-MiniLM-L12-v2",
        "embedding_dim": embeddings[0].shape[0] if embeddings.size > 0 else 0,
        "total_entities": len(entity_info),
        "clustered_entities": sum(1 for v in entity_info.values() if v['cluster_id']),
        "total_clusters": len(entity_clusters),
        "source_graph": graph_path
    }
    
    metadata_path = os.path.join(output_dir, 'embeddings_metadata.json')
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    
    print(f"- 메타데이터: {metadata_path}")
    
    return entity_info

In [ ]:
# 임베딩 생성 실행 (저장된 파일 사용)
clustered_graph_path = os.path.join(PATHS["clustered"], "final_clustered_graph.json")
aggregated_graph_path = os.path.join(PATHS["aggregated"], "aggregated_graph.json")

# 클러스터링된 그래프가 있는지 확인
if os.path.exists(clustered_graph_path):
    print("🎯 최종 클러스터링 그래프에 대한 임베딩 생성")
    entity_info = create_entity_embeddings_with_clusters(clustered_graph_path, PATHS["clustered"])
    
    # 클러스터 크기 분석
    print("\n📊 클러스터 크기 분석:")
    cluster_sizes = Counter()
    for info in entity_info.values():
        if info['cluster_id']:
            cluster_size = len(info['cluster_members']) + 1  # 자기 자신 포함
            cluster_sizes[cluster_size] += 1
    
    print("클러스터 크기별 분포:")
    for size, count in sorted(cluster_sizes.items()):
        print(f"  - {size}개 엔티티: {count}개 클러스터")
        
    # 가장 많은 연결을 가진 엔티티 (1-hop 기준)
    print("\n🌟 가장 많은 1-hop 연결을 가진 엔티티 (상위 10개):")
    connection_counts = []
    for entity, info in entity_info.items():
        if 'neighbors' in info:
            total_connections = len(info['neighbors']['outgoing']) + len(info['neighbors']['incoming'])
            connection_counts.append((entity, total_connections))
    
    connection_counts.sort(key=lambda x: x[1], reverse=True)
    for i, (entity, count) in enumerate(connection_counts[:10], 1):
        print(f"  {i:2d}. {entity}: {count}개 연결")
        
elif os.path.exists(aggregated_graph_path):
    print("🎯 통합 그래프에 대한 임베딩 생성 (클러스터링 없음)")
    entity_info = create_entity_embeddings_with_clusters(aggregated_graph_path, PATHS["aggregated"])
else:
    print("⚠️ 임베딩을 생성할 그래프 파일이 없습니다.")
    print(f"확인된 경로:")
    print(f"- 클러스터링 그래프: {clustered_graph_path} (존재: {os.path.exists(clustered_graph_path)})")
    print(f"- 통합 그래프: {aggregated_graph_path} (존재: {os.path.exists(aggregated_graph_path)})")